# Train 410M Score-Pool Mini-Universe Models on Colab

This runbook is the canonical from-scratch Colab workflow for
`tasks/TASK_train_410m_100m_color_filtered_books.md`. It trains the six original
selection policies plus five hard-pool ablations, evaluates them during training,
writes all metrics and figures to Google Drive, and produces a reproducible report.

The eleven base production runs are:

1. `random_positive_oracle_100k`
2. `random_pair_cascade_100k`
3. `random_union_control_100k`
4. `hard_positive_oracle_100k`
5. `hard_pair_cascade_100k`
6. `hard_union_control_100k`
7. `hard_pair_mid2_only_100k`
8. `hard_pair_mid4_only_100k`
9. `hard_dropout_embed_p000001_conservative_100k`
10. `hard_dropout_embed_p0005_conservative_100k`
11. `hard_dropout_embed_p001_conservative_100k`

`pair_mid4` removes original zero-based blocks `4-7`, the middle four blocks,
from both 12-layer scoring models before computing the direct CoLoR score.

Seeds 18 and 19 are generated for all eight hard-source methods. Together with
the unsuffixed seed-17 base runs, this gives three training seeds per hard-source
method and 27 production runs total. Repeat runs reuse the matching selected 100K
rows and vary only the training seed plus log/checkpoint identity. The two union
controls remain P2 controls: each samples 100K rows from its matched
positive/negative mini-universe without using CoLoR ranking.

The checked-in base sweep configs are used as templates. The Colab runtime
generates production configs with local Drive paths and explicit Books/C4 LM
evaluators before any expensive training starts. Do not edit configs by hand in
the notebook.

## Local Preconditions

The six original training datasets must already be built by:

```text
color-filter-ablation/scripts/18_build_score_pool_training_sets.py
```

Mirror this local folder to Drive before running Colab:

```text
color-filter-ablation/data/train-410m-score-pool-mini-universes
```

Expected Drive location:

```text
MyDrive/color-filter-ablation/data/train-410m-score-pool-mini-universes
```

Section 3 builds or validates the five additional hard-source datasets. It
discovers valid full-pool embedding-dropout summaries already on Drive and scores
only missing `p=1e-5`, `p=0.005`, or `p=0.01` sources with `K=8`. Raw scoring
shards and analyses persist on Drive, so reconnecting does not recompute validated
shards.

Production outputs intentionally use the `-2ep-full-eval` experiment suffix so
a fresh two-epoch rerun with Books/C4 learning curves cannot be confused with
earlier one-epoch, partial, or no-eval outputs.

The Books validation data is downloaded from the original CoLoR-Filter Hugging
Face model repo:

```text
hlzhang109/CoLoR-filter/downstream_data/books_val/books_val.npy
```

As of this runbook, `downstream_data` in that repo contains `books` and
`books_val`, but not a separate C4 validation memmap. The C4 evaluation below is
therefore a fixed, bounded proxy built from the public `allenai/c4` validation
split and labelled as `c4_val_proxy` in manifests, metrics, figures, and the
report. If an exact original C4 validation memmap becomes available, replace the
proxy path in Section 3 and keep the rest of the workflow unchanged.

## 0. Resource Assumptions

Use an A100 80GB runtime. A100 40GB may work with a smaller microbatch but has
less headroom for 410M-class training. Confirm Drive capacity for the three new pair-mid4 run checkpoints before
expanding the completed 24-run experiment.

Expected resources:

```text
GPU RAM:          A100 80GB preferred
System RAM:       Colab high-RAM recommended
Local scratch:    < 5GB for copied train/eval memmaps and logs
Drive data:       ~750MB for training sets, ~500MB for Books eval subset/cache
Drive outputs:    capacity for step390 and step780 checkpoints of remaining runs
Remote data:      Books eval from hlzhang109/CoLoR-filter; C4 proxy from allenai/c4
```

Training budget:

```text
unique rows per dataset:              100,000
sequence length:                      512
unique tokens per dataset:            51.2M
global batch size:                    256 sequences
tokens per optimizer step:            131,072
optimizer steps per epoch:            390
epochs per run:                       2
optimizer steps per run:              780
tokens per run:                       102,236,160
eleven base-run optimizer steps:      8,580
sixteen repeat-run optimizer steps:   12,480
all 27-run optimizer steps:           21,060
increment from completed 24-run grid: 2,340
```

At the measured approximately 47.5 minutes per run on an A100, the three new
pair-mid4 runs require about 2.4 sequential GPU-hours. Production evals run every 78
optimizer steps so the curves include the final step 780 checkpoint:

```text
eval steps per run: 78, 156, 234, 312, 390, 468, 546, 624, 702, 780
```

The production configs use `max_duration: 2ep`, not integer step count `764`.
OLMo stops at the end of a finite memmap epoch when `max_duration` is an integer,
so `764` only completed one 51.2M-token pass in earlier trials. Two epochs gives
the intended approximately 100M-token budget while preserving the 100K unique
training rows.

## 1. Runtime And Drive

One-time setup. Check the runtime before using Drive or installing packages:


In [ ]:
# PYTHON CELL
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select an A100 GPU runtime before continuing.")
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print("GPU:", gpu_name, f"{gpu_memory_gib:.1f} GiB")
if "A100" not in gpu_name:
    raise RuntimeError(f"This runbook requires an A100 runtime; Colab assigned {gpu_name}.")
!nvidia-smi


One-time setup. Mount Drive in its own cell:


In [ ]:
# PYTHON CELL
from google.colab import drive
drive.mount("/content/drive")


Safe to rerun. Define all stable paths:


In [ ]:
# PYTHON CELL
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/color-filter-ablation")
TRAIN_DATASET = "train-410m-score-pool-mini-universes"
EXPERIMENT = "train-410m-score-pool-mini-universes-2ep-full-eval"

TRAIN_DATA_DRIVE = DRIVE / "data" / TRAIN_DATASET
EVAL_DATA_DRIVE = DRIVE / "data" / "eval" / EXPERIMENT
CHECKPOINTS_DRIVE = DRIVE / "checkpoints" / EXPERIMENT
RESULTS_DRIVE = DRIVE / "results" / EXPERIMENT
REPORTS_DRIVE = DRIVE / "reports" / EXPERIMENT
FIGURES_DRIVE = REPORTS_DRIVE / "figures"
RUNTIME_CONFIG_DIR = Path("/content/score_pool_410m_runtime_configs")

EXTRA_TRAIN_RUN_IDS = [
    "hard_pair_mid2_only_100k",
    "hard_pair_mid4_only_100k",
    "hard_dropout_embed_p000001_conservative_100k",
    "hard_dropout_embed_p0005_conservative_100k",
    "hard_dropout_embed_p001_conservative_100k",
]
DROPOUT_LCB_SOURCE_DIR = DRIVE / "artifacts" / "score-pool-dropout-lcb-sources"
DROPOUT_LCB_SUMMARIES = {
    "hard_dropout_embed_p000001_conservative_100k": DROPOUT_LCB_SOURCE_DIR / "dropout_embed_p000001" / "color_distribution_summary.parquet",
    "hard_dropout_embed_p0005_conservative_100k": DROPOUT_LCB_SOURCE_DIR / "dropout_embed_p0005" / "color_distribution_summary.parquet",
    "hard_dropout_embed_p001_conservative_100k": DROPOUT_LCB_SOURCE_DIR / "dropout_embed_p001" / "color_distribution_summary.parquet",
}

for path in [EVAL_DATA_DRIVE, CHECKPOINTS_DRIVE, RESULTS_DRIVE, REPORTS_DRIVE, FIGURES_DRIVE, RUNTIME_CONFIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("training dataset:", TRAIN_DATASET)
print("experiment:", EXPERIMENT)
print("train data:", TRAIN_DATA_DRIVE)
print("dropout LCB summaries:", DROPOUT_LCB_SUMMARIES)
print("checkpoints:", CHECKPOINTS_DRIVE)
print("results:", RESULTS_DRIVE)
print("reports:", REPORTS_DRIVE)
print("figures:", FIGURES_DRIVE)


One-time setup. Check disk before any GPU work:


In [ ]:
# PYTHON CELL
!df -h /content /content/drive/MyDrive


If Drive has less than 50GB free, lower checkpoint retention in Section 5 before
starting production training.

## 2. Clone, Pin, And Install

Set the three revision variables to pushed commits before running this section.
`OLMO_PRODUCER_SHA` controls train-config generation and GPU training; changing it
invalidates producer artifacts. `OLMO_ANALYSIS_SHA` controls report/bundle code and
can change without recomputing completed training. `NOTEBOOK_REVISION` records the
committed notebook revision used to orchestrate this run. The runbook stops on
placeholders; never run a reproducibility job from an unpinned branch.


In [ ]:
# PYTHON CELL
ABLATION_PRODUCER_SHA = "4fc7d7ca972a6d6cbbb400a4c0f8f78e6af67bed"
OLMO_PRODUCER_SHA = "263856c6b3e88ce6023407ef3f19df08320ebe9c"
OLMO_ANALYSIS_SHA = "aa42e1783ef054da262ceecdc24578d1ac85762a"
NOTEBOOK_REVISION = "aa42e1783ef054da262ceecdc24578d1ac85762a"


Safe to rerun. Clone and assert exact commits:


In [ ]:
# PYTHON CELL
import subprocess
from pathlib import Path

repos = [
    {
        "name": "CoLoR-ablation producer",
        "path": Path("/content/CoLoR-ablation"),
        "repo": "https://github.com/myazdani/CoLoR-ablation.git",
        "sha": ABLATION_PRODUCER_SHA,
        "placeholder": "REPLACE_WITH_PUSHED_COLOR_ABLATION_COMMIT_SHA",
    },
    {
        "name": "color-filter-olmo producer",
        "path": Path("/content/color-filter-olmo"),
        "repo": "https://github.com/myazdani/color-filter-olmo.git",
        "sha": OLMO_PRODUCER_SHA,
        "placeholder": "REPLACE_WITH_PUSHED_COLOR_OLMO_PRODUCER_COMMIT_SHA",
    },
    {
        "name": "color-filter-olmo analysis",
        "path": Path("/content/color-filter-olmo-analysis"),
        "repo": "https://github.com/myazdani/color-filter-olmo.git",
        "sha": OLMO_ANALYSIS_SHA,
        "placeholder": "REPLACE_WITH_PUSHED_COLOR_OLMO_ANALYSIS_COMMIT_SHA",
    },
]

def run(*args, cwd=None):
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

def out(*args, cwd=None):
    return subprocess.check_output([str(arg) for arg in args], cwd=cwd, text=True).strip()

for item in repos:
    if item["sha"] == item["placeholder"]:
        raise RuntimeError(f"Set {item['name']} SHA before running this cell.")
    if not item["path"].exists():
        run("git", "clone", item["repo"], item["path"])
    elif (item["path"] / ".git").is_dir():
        run("git", "-C", item["path"], "fetch", "origin")
    else:
        raise RuntimeError(f"{item['path']} exists but is not a git checkout")
    run("git", "-C", item["path"], "checkout", item["sha"])
    actual = out("git", "-C", item["path"], "rev-parse", "HEAD")
    if actual != item["sha"]:
        raise RuntimeError(f"{item['name']} SHA mismatch: expected {item['sha']}, got {actual}")
    print(item["name"], actual)

if NOTEBOOK_REVISION == "REPLACE_WITH_PUSHED_NOTEBOOK_COMMIT_SHA":
    raise RuntimeError("Set NOTEBOOK_REVISION to the pushed commit containing this notebook.")
notebook_path_in_repo = "notebooks/train_410m_score_pool_mini_universes_colab.ipynb"
subprocess.run(
    ["git", "-C", "/content/color-filter-olmo", "cat-file", "-e", f"{NOTEBOOK_REVISION}:{notebook_path_in_repo}"],
    check=True,
)
print("notebook revision contains:", notebook_path_in_repo, NOTEBOOK_REVISION)


One-time setup. Install a Colab overlay only:


In [ ]:
# PYTHON CELL
import subprocess
import torch

torch_before = (torch.__version__, torch.version.cuda)
overlay = [
    "omegaconf==2.3.0",
    "cached_path==1.8.10",
    "boto3",
    "google-cloud-storage",
    "torchmetrics",
    "wandb",
    "datasets",
    "huggingface_hub",
    "transformers",
    "markdown",
]
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *overlay],
    check=True,
)
import torch as torch_after
if (torch_after.__version__, torch_after.version.cuda) != torch_before:
    raise RuntimeError(
        f"Colab overlay changed torch from {torch_before} to "
        f"{(torch_after.__version__, torch_after.version.cuda)}; restart with a clean runtime."
    )
pip_check = subprocess.run(
    ["python", "-m", "pip", "check"],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
if pip_check.returncode:
    print("pip check reported preinstalled-environment conflicts; continuing to the required import gate:")
    print(pip_check.stdout)
else:
    print("pip check passed")
print("overlay installed without changing torch:", torch_before)


Safe to rerun. Import check:


In [ ]:
# PYTHON CELL
import importlib
import torch

for module_name in [
    "numpy",
    "pandas",
    "pyarrow",
    "yaml",
    "omegaconf",
    "cached_path",
    "torchmetrics",
    "datasets",
    "huggingface_hub",
    "transformers",
    "matplotlib",
]:
    importlib.import_module(module_name)

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0))


Safe to rerun. Record the current runtime identity. A GPU change does not invalidate completed training shards, but it does invalidate the previous microbatch benchmark; rerun Section 6 before starting or resuming production on a different GPU class.


In [ ]:
# PYTHON CELL
import hashlib
import json
import platform
import torch

runtime_identity = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_total_memory_bytes": int(torch.cuda.get_device_properties(0).total_memory),
}
runtime_fingerprint = hashlib.sha256(
    json.dumps(runtime_identity, sort_keys=True).encode("utf-8")
).hexdigest()
RUNTIME_IDENTITY_PATH = RESULTS_DRIVE / "runtime_identity.json"
if RUNTIME_IDENTITY_PATH.exists():
    previous = json.loads(RUNTIME_IDENTITY_PATH.read_text())
    if previous.get("fingerprint") != runtime_fingerprint:
        print("runtime changed since the previous session; rerun the bounded microbatch benchmark")
        print("stored:", previous.get("runtime_identity"))
        print("current:", runtime_identity)

runtime_context = {
    "runtime_identity": runtime_identity,
    "fingerprint": runtime_fingerprint,
    "ablation_producer_sha": ABLATION_PRODUCER_SHA,
    "olmo_producer_sha": OLMO_PRODUCER_SHA,
    "olmo_analysis_sha": OLMO_ANALYSIS_SHA,
    "notebook_revision": NOTEBOOK_REVISION,
}
RUNTIME_IDENTITY_PATH.write_text(json.dumps(runtime_context, indent=2, sort_keys=True) + "\n")
print("runtime fingerprint:", runtime_fingerprint)
print("runtime context:", RUNTIME_IDENTITY_PATH)


Capability probe. Verify required files exist at the pinned SHAs:


In [ ]:
# PYTHON CELL
from pathlib import Path
import subprocess

OLMO_PRODUCER_DIR = Path("/content/color-filter-olmo")
OLMO_ANALYSIS_DIR = Path("/content/color-filter-olmo-analysis")

required_paths = [
    Path("/content/CoLoR-ablation/scripts/18_build_score_pool_training_sets.py"),
    Path("/content/CoLoR-ablation/scripts/21_ensure_score_pool_training_sets.py"),
    OLMO_PRODUCER_DIR / "scripts/build_score_pool_extra_training_sets.py",
    OLMO_PRODUCER_DIR / "scripts/score_pool_410m_colab.py",
    OLMO_PRODUCER_DIR / "scripts/targeted_dropout_colab_helpers.py",
    OLMO_PRODUCER_DIR / "scripts/21_dropout_uncertainty_metrics.py",
    OLMO_PRODUCER_DIR / "scripts/22_dropout_strategy_sweep.py",
    OLMO_PRODUCER_DIR / "scripts/train.py",
    OLMO_ANALYSIS_DIR / "scripts/score_pool_410m_report.py",
    OLMO_PRODUCER_DIR / "configs/sweeps/score-targeted-dropout-uncertainty.yaml",
    OLMO_PRODUCER_DIR / "configs/sweeps/score-pool-410m-100m-random-positive-oracle.yaml",
    OLMO_PRODUCER_DIR / "configs/sweeps/score-pool-410m-100m-random-pair-cascade.yaml",
    OLMO_PRODUCER_DIR / "configs/sweeps/score-pool-410m-100m-random-union-control.yaml",
    OLMO_PRODUCER_DIR / "configs/sweeps/score-pool-410m-100m-hard-positive-oracle.yaml",
    OLMO_PRODUCER_DIR / "configs/sweeps/score-pool-410m-100m-hard-pair-cascade.yaml",
    OLMO_PRODUCER_DIR / "configs/sweeps/score-pool-410m-100m-hard-union-control.yaml",
]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(required_path)
    print("ok:", required_path)

for script in [path for path in required_paths if path.suffix == ".py"]:
    subprocess.run(["python", "-m", "py_compile", str(script)], check=True)
print("helpers compile")


Safe to rerun. Import and fingerprint the pinned Colab orchestration helper. This explicit reload prevents a reused runtime from retaining an older module.


In [ ]:
# PYTHON CELL
import importlib
import sys

COLAB_HELPER_DIR = OLMO_PRODUCER_DIR / "scripts"
if str(COLAB_HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(COLAB_HELPER_DIR))
import score_pool_410m_colab as SCORE_POOL_COLAB
SCORE_POOL_COLAB = importlib.reload(SCORE_POOL_COLAB)
COLAB_HELPER_PATH = Path(SCORE_POOL_COLAB.__file__).resolve()
assert COLAB_HELPER_PATH == (COLAB_HELPER_DIR / "score_pool_410m_colab.py").resolve()
print("helper:", COLAB_HELPER_PATH)
print("helper sha256:", SCORE_POOL_COLAB.source_sha256(COLAB_HELPER_PATH))
print("producer SHA:", OLMO_PRODUCER_SHA)
print("analysis SHA:", OLMO_ANALYSIS_SHA)
print("notebook revision:", NOTEBOOK_REVISION)


## 3. Prepare Dropout Sources, Train Data, And Eval Data

Safe to rerun. This section validates existing full-pool embedding-dropout
summaries and scores only missing `K=8` sources for `p=1e-5`, `p=0.005`, and
`p=0.01`. Its bounded smoke and microbatch benchmark run before full scoring.
Raw shards persist under
`results/score-pool-embedding-dropout-sources-500k/`; reconnecting and rerunning
skips validated shards. The measured source-scoring ETA is printed before the
long loop.

After all three summaries pass exact 500K alignment and rate checks, this
section stages stable aliases, builds the direct-pair and LCB datasets, and
copies all training memmaps to local scratch to avoid Drive read latency.


In [ ]:
# PYTHON CELL
from pathlib import Path
import subprocess

ensure_train_sets = Path("/content/CoLoR-ablation/scripts/21_ensure_score_pool_training_sets.py")
extra_builder = OLMO_PRODUCER_DIR / "scripts/build_score_pool_extra_training_sets.py"
subprocess.run([
    "python", str(ensure_train_sets),
    "--drive-root", str(DRIVE),
    "--output-dir", str(TRAIN_DATA_DRIVE),
    "--rebuild-if-missing",
], check=True)

dropout_sources = SCORE_POOL_COLAB.ensure_dropout_lcb_sources(
    SCORE_POOL_COLAB.DropoutSourceContext(
        drive_root=DRIVE,
        olmo_dir=OLMO_PRODUCER_DIR,
        staging_paths=DROPOUT_LCB_SUMMARIES,
        producer_sha=OLMO_PRODUCER_SHA,
        notebook_revision=NOTEBOOK_REVISION,
        runtime_identity=runtime_identity,
    )
)
print("validated dropout sources:", dropout_sources)

extra_cmd = [
    "python", str(extra_builder),
    "--tokens", str(DRIVE / "data" / "score_pool_tokens_official_500k.npy"),
    "--metadata", str(DRIVE / "data" / "score_pool_meta_official_500k.parquet"),
    "--pair-mid2-scores", str(DRIVE / "results" / "score-pool-robustness-official-500k" / "scores_pair_mid2.parquet"),
    "--pair-mid4-scores", str(DRIVE / "results" / "score-pool-robustness-official-500k" / "scores_pair_mid4.parquet"),
    "--output-dir", str(TRAIN_DATA_DRIVE),
]
for run_id, summary_path in dropout_sources.items():
    extra_cmd.extend(["--dropout-selection", f"{run_id}={summary_path}"])
subprocess.run(extra_cmd, check=True)

LOCAL_TRAIN_DATA = Path("/content/score_pool_train_data")
SCORE_POOL_COLAB.stage_local_training_data(TRAIN_DATA_DRIVE, LOCAL_TRAIN_DATA)
!du -sh /content/score_pool_train_data


Safe to rerun. Validate the local copy with the same versioned helper:


In [ ]:
# PYTHON CELL
import json

BASE_TRAIN_RUN_IDS = [
    "random_positive_oracle_100k", "random_pair_cascade_100k", "random_union_control_100k",
    "hard_positive_oracle_100k", "hard_pair_cascade_100k", "hard_union_control_100k",
]
ALL_BASE_TRAIN_RUN_IDS = BASE_TRAIN_RUN_IDS + EXTRA_TRAIN_RUN_IDS
for run_id in ALL_BASE_TRAIN_RUN_IDS:
    run_dir = LOCAL_TRAIN_DATA / run_id
    manifest_path = run_dir / "manifest.json"
    token_path = run_dir / "train_tokens.npy"
    assert manifest_path.is_file(), manifest_path
    assert token_path.is_file(), token_path
    manifest = json.loads(manifest_path.read_text())
    assert manifest["actual_unique_rows"] == 100_000, (run_id, manifest)
    assert token_path.stat().st_size == 100_000 * 512 * 2, (run_id, token_path.stat().st_size)
    print("validated:", run_id, manifest["selection_policy"])


Expected `m=1.5` P0 true-positive rates from the previous local build:

```text
random_positive_oracle_100k: 1.00000
hard_positive_oracle_100k:   1.00000
random_pair_cascade_100k:    0.93338
hard_pair_cascade_100k:      0.66184
```

The union controls are random 100K samples from balanced 200K-row universes, so
their true-positive rates should be near 0.5 but are not asserted exactly.

Safe to rerun. Create bounded, aligned LM eval memmaps. Books uses the original
CoLoR-Filter Books validation tokens. C4 uses a fixed public validation proxy
because the original repo does not expose `downstream_data/c4_val`.


In [ ]:
# PYTHON CELL
from pathlib import Path

EVAL_SEQUENCES = 4096
SEQ_LEN = 512
EVAL_SUBSET_NUM_BATCHES = 100
DEVICE_EVAL_BATCH_SIZE = 16

BOOKS_EVAL = EVAL_DATA_DRIVE / f"books_val_{EVAL_SEQUENCES}x{SEQ_LEN}_uint16.npy"
C4_EVAL = EVAL_DATA_DRIVE / f"c4_val_proxy_{EVAL_SEQUENCES}x{SEQ_LEN}_uint16.npy"
EVAL_MANIFEST = EVAL_DATA_DRIVE / "eval_manifest.json"

SCORE_POOL_COLAB.write_eval_data(
    books_eval=BOOKS_EVAL,
    c4_eval=C4_EVAL,
    eval_manifest=EVAL_MANIFEST,
    eval_sequences=EVAL_SEQUENCES,
    sequence_length=SEQ_LEN,
    device_eval_batch_size=DEVICE_EVAL_BATCH_SIZE,
    eval_subset_num_batches=EVAL_SUBSET_NUM_BATCHES,
)


Safe to rerun. Validate eval memmaps:


In [ ]:
# PYTHON CELL
SCORE_POOL_COLAB.validate_eval_data(
    [BOOKS_EVAL, C4_EVAL],
    eval_sequences=EVAL_SEQUENCES,
    sequence_length=SEQ_LEN,
)


## 4. Generate Runtime Configs

Safe to rerun. Generate production configs with explicit evals and Drive output
paths. These generated configs are the only configs used for production runs.


In [ ]:
# PYTHON CELL
import os
import sys

OLMO_DIR = OLMO_PRODUCER_DIR
assert (OLMO_DIR / "scripts/train.py").exists(), OLMO_DIR
if str(OLMO_DIR) not in sys.path:
    sys.path.insert(0, str(OLMO_DIR))
os.environ.update({
    "SCORE_POOL_TRAIN_DATA_DIR": str(LOCAL_TRAIN_DATA),
    "SCORE_POOL_CHECKPOINTS_DIR": str(CHECKPOINTS_DRIVE),
    "PYTHONUNBUFFERED": "1",
})

template = OLMO_DIR / "configs/sweeps/score-pool-410m-100m-hard-pair-cascade.yaml"
config_map = {
    "random_positive_oracle_100k": OLMO_DIR / "configs/sweeps/score-pool-410m-100m-random-positive-oracle.yaml",
    "random_pair_cascade_100k": OLMO_DIR / "configs/sweeps/score-pool-410m-100m-random-pair-cascade.yaml",
    "random_union_control_100k": OLMO_DIR / "configs/sweeps/score-pool-410m-100m-random-union-control.yaml",
    "hard_positive_oracle_100k": OLMO_DIR / "configs/sweeps/score-pool-410m-100m-hard-positive-oracle.yaml",
    "hard_pair_cascade_100k": template,
    "hard_union_control_100k": OLMO_DIR / "configs/sweeps/score-pool-410m-100m-hard-union-control.yaml",
    "hard_pair_mid2_only_100k": template,
    "hard_pair_mid4_only_100k": template,
    "hard_dropout_embed_p000001_conservative_100k": template,
    "hard_dropout_embed_p0005_conservative_100k": template,
    "hard_dropout_embed_p001_conservative_100k": template,
}
runtime_config_map = SCORE_POOL_COLAB.generate_runtime_configs(
    config_map=config_map, train_data_dir=LOCAL_TRAIN_DATA,
    checkpoints_dir=CHECKPOINTS_DRIVE, runtime_config_dir=RUNTIME_CONFIG_DIR,
    books_eval=BOOKS_EVAL, c4_eval=C4_EVAL,
    device_eval_batch_size=DEVICE_EVAL_BATCH_SIZE,
    eval_subset_num_batches=EVAL_SUBSET_NUM_BATCHES,
)


Hard-source seed repeats. Edit `HARD_SOURCE_REPEAT_SEEDS` to add or remove
training seeds; set it to `[]` to train only the eleven base runs. Each repeat
reuses its method's selected 100K rows and varies only the training seed plus
log/checkpoint identity.


In [ ]:
# PYTHON CELL
HARD_SOURCE_REPEAT_SEEDS = [18, 19]
HARD_SOURCE_SEED_BASE_RUNS = [
    "hard_positive_oracle_100k",
    "hard_pair_cascade_100k",
    "hard_union_control_100k",
    "hard_pair_mid2_only_100k",
    "hard_pair_mid4_only_100k",
    "hard_dropout_embed_p000001_conservative_100k",
    "hard_dropout_embed_p0005_conservative_100k",
    "hard_dropout_embed_p001_conservative_100k",
]
base_production_order = list(runtime_config_map)
hard_source_seed_run_ids = SCORE_POOL_COLAB.generate_seed_configs(
    runtime_config_map=runtime_config_map,
    seeds=HARD_SOURCE_REPEAT_SEEDS,
    base_run_ids=HARD_SOURCE_SEED_BASE_RUNS,
    checkpoints_dir=CHECKPOINTS_DRIVE,
    runtime_config_dir=RUNTIME_CONFIG_DIR,
)
production_order = list(runtime_config_map)
EXPECTED_EVAL_POINTS = 10
expected_seed_run_count = len(HARD_SOURCE_REPEAT_SEEDS) * len(HARD_SOURCE_SEED_BASE_RUNS)
assert len(base_production_order) == 11, base_production_order
assert len(hard_source_seed_run_ids) == expected_seed_run_count, hard_source_seed_run_ids
assert len(production_order) == len(base_production_order) + expected_seed_run_count
assert len(production_order) == 27, production_order
assert len(production_order) == len(set(production_order))
print("hard-source seed repeats:", hard_source_seed_run_ids)
print(f"report run order: {len(production_order)} runs", production_order)


Safe to rerun. Load generated configs through OLMo and print parameter count:


In [ ]:
# PYTHON CELL
import os
import sys
from pathlib import Path

os.chdir(OLMO_DIR)
if str(OLMO_DIR) not in sys.path:
    sys.path.insert(0, str(OLMO_DIR))

from olmo.config import TrainConfig
from olmo.model import OLMo

for run_id, cfg_path in runtime_config_map.items():
    cfg = TrainConfig.load(str(cfg_path))
    assert cfg.max_duration == "2ep"
    assert cfg.global_train_batch_size == 256
    assert cfg.model.max_sequence_length == 512
    assert cfg.data.memmap_dtype == "uint16"
    assert len(cfg.evaluators) == 2
    assert cfg.eval_interval == 78
    assert Path(cfg.data.paths[0]).exists(), cfg.data.paths[0]
    for ev in cfg.evaluators:
        assert ev.type.value == "lm"
        assert ev.subset_num_batches == EVAL_SUBSET_NUM_BATCHES
        assert Path(ev.data.paths[0]).exists(), ev.data.paths[0]
    print(run_id, cfg.run_name, cfg.data.paths[0], [ev.label for ev in cfg.evaluators])

cfg = TrainConfig.load(str(runtime_config_map["random_positive_oracle_100k"]))
model = OLMo(cfg.model)
print("total parameters:", f"{model.num_params():,}")
print("non-embedding parameters:", f"{model.num_params(include_embedding=False):,}")
del model


Safe to rerun. Define the subprocess helper used by smoke tests, microbatch
tuning, and production training:


In [ ]:
# PYTHON CELL
from pathlib import Path

def run_logged(cmd, log_path: Path, cwd: Path = OLMO_DIR, append: bool = False) -> None:
    SCORE_POOL_COLAB.run_logged(
        cmd,
        log_path,
        cwd=cwd,
        pythonpath=OLMO_DIR,
        append=append,
        heartbeat_seconds=30,
    )


## 5. Cheap GPU Gate

Benchmark only. This smoke test trains five steps on each of the eleven base data
policies and runs a two-batch Books/C4 eval at step five. That is 55 global
training batches, 7,208,960 training tokens, and 44 eval batches total. Expect
roughly 10-25 minutes on an A100, including process startup and compilation.
Smoke outputs are isolated under `smoke/` and can be deleted after inspection.


In [ ]:
# PYTHON CELL
SMOKE_DIR = DRIVE / "smoke" / EXPERIMENT
SMOKE_CONFIG_DIR = Path("/content/score_pool_410m_smoke_configs")
smoke_run_ids = list(config_map)
smoke_config_map = SCORE_POOL_COLAB.generate_smoke_configs(
    runtime_config_map=runtime_config_map,
    run_ids=smoke_run_ids,
    smoke_dir=SMOKE_DIR,
    smoke_config_dir=SMOKE_CONFIG_DIR,
)


In [ ]:
# PYTHON CELL
for run_id, cfg_path in smoke_config_map.items():
    log_path = SMOKE_DIR / f"{run_id}.log"
    run_logged([
        "torchrun",
        "--standalone",
        "--nproc_per_node=1",
        "scripts/train.py",
        str(cfg_path),
        "--device_train_microbatch_size=8",
        "--save_overwrite=true",
    ], log_path)


Safe to rerun. Verify smoke logs contain train and eval metrics:


In [ ]:
# PYTHON CELL
SCORE_POOL_COLAB.validate_smoke_logs(smoke_config_map, SMOKE_DIR)


Cleanup. Delete only isolated smoke artifacts:


In [ ]:
# PYTHON CELL
import shutil

print("cleanup target:", SMOKE_DIR)
if "/smoke/" not in str(SMOKE_DIR):
    raise RuntimeError(f"Refusing to delete non-smoke path: {SMOKE_DIR}")
shutil.rmtree(SMOKE_DIR)
print("deleted:", SMOKE_DIR)


## 6. Microbatch Tuning

Benchmark only. Each candidate trains exactly 20 global batches: 5,120 rows and
2,621,440 tokens, with evaluators disabled and outputs isolated by runtime
fingerprint. The production global/device batch is fixed at 256 to preserve
training semantics, so valid microbatches are `32, 64, 128, 256`. The ladder
stops at the first OOM, at 65-72GB peak memory, or at the device-batch ceiling.
Selection uses the fastest stable candidate with at least two throughput
observations. Expect roughly 5-15 minutes on an A100 for the full ladder.


In [ ]:
# PYTHON CELL
MICROBATCH_CANDIDATES = [32, 64, 128, 256]
MICROBATCH_SOFT_STOP_MB = 65_000
MICROBATCH_PEAK_LIMIT_MB = 72_000
BENCHMARK_ID = runtime_fingerprint[:12]
MICROBATCH_TEST_CONFIG_DIR = Path("/content/score_pool_410m_microbatch_configs")
BENCHMARK_DIR = RESULTS_DRIVE / "benchmarks" / BENCHMARK_ID
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
print("benchmark rows per candidate: 5,120")
print("benchmark tokens per candidate: 2,621,440")
microbatch_config_map = SCORE_POOL_COLAB.generate_microbatch_configs(
    base_config_path=runtime_config_map["random_positive_oracle_100k"],
    candidates=MICROBATCH_CANDIDATES,
    benchmark_id=BENCHMARK_ID,
    checkpoints_dir=CHECKPOINTS_DRIVE,
    config_dir=MICROBATCH_TEST_CONFIG_DIR,
)


In [ ]:
# PYTHON CELL
microbatch_failures = {}
benchmark_run_count = len(base_production_order) + len(globals().get("hard_source_seed_run_ids", []))
for microbatch, cfg_path in microbatch_config_map.items():
    log_path = BENCHMARK_DIR / f"microbatch_{microbatch}.log"
    try:
        run_logged([
            "torchrun", "--standalone", "--nproc_per_node=1", "scripts/train.py",
            str(cfg_path), f"--device_train_microbatch_size={microbatch}",
            "--save_overwrite=true",
        ], log_path)
    except RuntimeError as exc:
        microbatch_failures[microbatch] = str(exc)
        print(f"microbatch {microbatch} failed; stopping the ladder")
        print("If this was an OOM, restart the runtime before production training.")
        break
    observed, _ = SCORE_POOL_COLAB.parse_microbatch_results(
        candidates=[microbatch], benchmark_dir=BENCHMARK_DIR,
        peak_limit_mb=MICROBATCH_PEAK_LIMIT_MB, total_runs=benchmark_run_count,
    )
    if observed and float(observed[0]["peak_mb"]) >= MICROBATCH_SOFT_STOP_MB:
        print(f"reached memory headroom at microbatch {microbatch}; stopping the ladder")
        break


Safe to rerun. Estimate production time and select the fastest completed
microbatch with at least two throughput observations and memory headroom:


In [ ]:
# PYTHON CELL
microbatch_results, recommended_microbatch = SCORE_POOL_COLAB.parse_microbatch_results(
    candidates=MICROBATCH_CANDIDATES,
    benchmark_dir=BENCHMARK_DIR,
    peak_limit_mb=MICROBATCH_PEAK_LIMIT_MB,
    total_runs=len(base_production_order) + len(globals().get("hard_source_seed_run_ids", [])),
)
print("benchmark runtime fingerprint:", runtime_fingerprint)
print("recommended_microbatch:", recommended_microbatch)


Set `MICROBATCH` from the benchmark. If the benchmark cell was skipped, use the
known-stable fallback `32`.


In [ ]:
# PYTHON CELL
MICROBATCH = globals().get("recommended_microbatch", 32)
print("MICROBATCH:", MICROBATCH)


## 7. Full Resumable Training Runs

Full run. Train the eleven base models plus seeds 18 and 19 for all eight hard-source
methods. With the unsuffixed seed-17 runs, this produces three training seeds per
hard-source method and 27 runs total. The helper skips logs that contain `Training
complete` and the expected eval curves, so expanding the completed 24-run experiment
launches only the three newly configured pair-mid4 runs. If interrupted before completion, it
appends to the existing log and resumes from the latest checkpoint so partial learning
curves are preserved.


In [ ]:
# PYTHON CELL
def production_log_status(log_path: Path) -> dict:
    return SCORE_POOL_COLAB.production_log_status(log_path, EXPECTED_EVAL_POINTS)

def run_training(run_id: str) -> None:
    SCORE_POOL_COLAB.run_training(
        run_id=run_id,
        config_path=runtime_config_map[run_id],
        checkpoint_dir=CHECKPOINTS_DRIVE / run_id,
        log_path=RESULTS_DRIVE / f"{run_id}.log",
        olmo_dir=OLMO_DIR,
        microbatch=MICROBATCH,
        expected_eval_points=EXPECTED_EVAL_POINTS,
    )

statuses = {run_id: production_log_status(RESULTS_DRIVE / f"{run_id}.log") for run_id in production_order}
completed = sum(bool(status["training_complete"] and status["has_required_eval_curve"]) for status in statuses.values())
benchmark = next((item for item in globals().get("microbatch_results", []) if item["microbatch"] == MICROBATCH), None)
print(f"production: {completed}/{len(production_order)} complete; {len(production_order) - completed} remain")
if benchmark:
    print(f"ETA: remaining runs * {benchmark['eta_per_run_hours']:.2f}h/run; output={CHECKPOINTS_DRIVE}")
else:
    print("ETA unavailable: run Section 6 before production")
for run_id in production_order:
    run_training(run_id)


Do not use `--save_overwrite=true` in production unless intentionally replacing
a run.

## 8. Resume After Disconnect

After reconnect, rerun Sections 1, 2, 3, 4, and the final `MICROBATCH`
selection cell from Section 6. If `/content/score_pool_train_data` still exists
and Section 3 validation passes, the local data copy can be skipped.

Safe to rerun. Check status:


In [ ]:
# PYTHON CELL
print("MICROBATCH:", globals().get("MICROBATCH", "not set"))
for run_id in production_order:
    save_path = CHECKPOINTS_DRIVE / run_id
    log_path = RESULTS_DRIVE / f"{run_id}.log"
    print("===", run_id, "===")
    print("log:", log_path.exists(), log_path.stat().st_size if log_path.exists() else 0)
    if log_path.exists():
        tail = log_path.read_text(errors="ignore")[-2000:]
        print("complete:", "Training complete" in tail)
        if "production_log_status" in globals():
            print("eval status:", production_log_status(log_path))
    if save_path.exists():
        print("checkpoints:", [p.name for p in sorted(save_path.glob("step*"))])
    else:
        print("checkpoint dir missing")


Resume one run. Set `RUN_TO_RESUME` to any configured run ID printed in Section 7:


In [ ]:
# PYTHON CELL
RUN_TO_RESUME = "hard_pair_cascade_seed18_100k"
if RUN_TO_RESUME not in runtime_config_map:
    raise KeyError(f"{RUN_TO_RESUME} is not configured. Available runs: {list(runtime_config_map)}")
run_training(RUN_TO_RESUME)


## 9. Metrics, Figures, And Report

Safe to rerun after any completed production logs exist. This calls the checked-in
report helper so metrics parsing, figures, Markdown, HTML, and acceptance logic
stay versioned with the repo instead of living as bulky notebook code. It writes
all artifacts directly to Drive.


In [ ]:
# PYTHON CELL
report_run_ids = list(globals().get("production_order", base_production_order))

def assert_eval_curves_ready() -> None:
    SCORE_POOL_COLAB.assert_eval_curves_ready(report_run_ids, RESULTS_DRIVE, EXPECTED_EVAL_POINTS)

def run_report_helper(check_only: bool = False) -> None:
    cmd = [
        "python", "-u", "scripts/score_pool_410m_report.py",
        "--train-data-dir", str(LOCAL_TRAIN_DATA),
        "--results-dir", str(RESULTS_DRIVE),
        "--reports-dir", str(REPORTS_DRIVE),
        "--checkpoints-dir", str(CHECKPOINTS_DRIVE),
        "--eval-manifest", str(EVAL_MANIFEST),
        "--experiment", EXPERIMENT,
        "--runtime-config-dir", str(RUNTIME_CONFIG_DIR),
        "--ablation-sha", ABLATION_PRODUCER_SHA,
        "--olmo-sha", OLMO_PRODUCER_SHA,
        "--sequence-length", str(SEQ_LEN),
        "--eval-subset-num-batches", str(EVAL_SUBSET_NUM_BATCHES),
        "--eval-interval", "78",
        "--device-eval-batch-size", str(DEVICE_EVAL_BATCH_SIZE),
    ]
    for run_id in report_run_ids:
        cmd.extend(["--run-id", run_id])
    if check_only:
        cmd.append("--check-only")
    run_logged(
        cmd,
        RESULTS_DRIVE / "report_generation.log",
        cwd=OLMO_ANALYSIS_DIR,
        append=(RESULTS_DRIVE / "report_generation.log").exists(),
    )

assert_eval_curves_ready()
run_report_helper()


Optional quick previews:


In [ ]:
# PYTHON CELL
import pandas as pd

display(pd.read_csv(RESULTS_DRIVE / "selection_diagnostics.csv"))
display(pd.read_csv(RESULTS_DRIVE / "throughput_comparison.csv"))
print("report:", REPORTS_DRIVE / "report.md")
print("html:", REPORTS_DRIVE / "report.html")
print("figures:", sorted(p.name for p in FIGURES_DRIVE.glob("*.png")))


## 10. Outputs To Bring Back Locally

The durable outputs are under:

```text
MyDrive/color-filter-ablation/results/train-410m-score-pool-mini-universes-2ep-full-eval
MyDrive/color-filter-ablation/reports/train-410m-score-pool-mini-universes-2ep-full-eval
MyDrive/color-filter-ablation/checkpoints/train-410m-score-pool-mini-universes-2ep-full-eval
```

The required report figures are:

```text
figures/train_loss_by_run.png
figures/eval_loss_books_by_run.png
figures/eval_loss_books_hard_oracle_vs_cascade_seeds.png
figures/eval_loss_books_hard_source_seed_ci.png
figures/eval_loss_c4_by_run.png
figures/tokens_per_second_by_run.png
figures/selection_full_score_distributions.png
figures/selection_pair_mid2_score_distributions.png
figures/selection_pair_mid4_score_distributions.png
figures/selected_set_overlap_heatmap.png
```

The seed-level aggregate used by the confidence-interval figure is written to
`results/.../hard_source_seed_eval_summary.csv`. Safe to rerun after Section 9.
This builds a lightweight zip for local figure regeneration. It includes logs,
generated metrics, reports, figures, runtime configs, eval manifest, and train-set
metadata, but intentionally excludes checkpoint weights and `train_tokens.npy`.


In [ ]:
# PYTHON CELL
ZIP_PATH = Path(f"/content/score_pool_410m_{EXPERIMENT}_figure_bundle.zip")
BUNDLE_DRIVE_PATH = RESULTS_DRIVE / ZIP_PATH.name
bundle_run_ids = tuple(globals().get("report_run_ids", globals().get("production_order", base_production_order)))
bundle_context = SCORE_POOL_COLAB.BundleContext(
    experiment=EXPERIMENT,
    train_dataset=TRAIN_DATASET,
    run_ids=bundle_run_ids,
    train_data_drive=TRAIN_DATA_DRIVE,
    results_drive=RESULTS_DRIVE,
    reports_drive=REPORTS_DRIVE,
    runtime_config_dir=RUNTIME_CONFIG_DIR,
    eval_manifest=EVAL_MANIFEST,
    analysis_helper=OLMO_ANALYSIS_DIR / "scripts/score_pool_410m_report.py",
    orchestration_helper=COLAB_HELPER_PATH,
    ablation_producer_sha=ABLATION_PRODUCER_SHA,
    olmo_producer_sha=OLMO_PRODUCER_SHA,
    olmo_analysis_sha=OLMO_ANALYSIS_SHA,
    notebook_revision=NOTEBOOK_REVISION,
)
SCORE_POOL_COLAB.build_bundle(bundle_context, ZIP_PATH, BUNDLE_DRIVE_PATH)


## 11. Output Review And Acceptance Checks

Safe to rerun. Run this after Section 9. It verifies the metrics CSVs, report
files, all required figures, final eval curves, final step780 checkpoints, and
100K-row selection diagnostics.


In [ ]:
# PYTHON CELL
assert_eval_curves_ready()
run_report_helper(check_only=True)


## 12. Stop Rules And Interpretation

Pause before launching later runs if:

- the smoke cell fails on either evaluator;
- a production run produces non-finite train or eval losses;
- Books validation is missing from logs after a completed run;
- Drive has insufficient space for the remaining checkpoints;
- an LCB source summary fails its alignment, target, rate, or `K=8` validation.

The direct pair-only variants isolate scorer depth and the value of the full rerank:

- `hard_pair_mid2_only_100k` vs `hard_pair_cascade_100k`;
- `hard_pair_mid4_only_100k` vs `hard_pair_mid2_only_100k` and the full cascade.

The `pair_mid4` source must report zero-based blocks `4-7` removed from both the
conditional and marginal 12-layer scoring models; Section 3 rejects any other provenance.

The three LCB variants hold the hard 200K-row universe and 100K selection budget fixed,
changing only the embedding-dropout rate. They select the lowest
`mean_color + 1.0 * std_color`, which is the lower-confidence bound on utility under
the lower-is-better CoLoR convention. Compare each against the hard positive oracle,
hard-union control, direct pair-mid2 and pair-mid4 selectors, and full-rerank cascade. The C4 proxy is
a secondary general-domain sanity check, not the primary target metric.


## 13. Download Verified Bundle

Safe to rerun. This final cell only verifies and downloads the existing bundle; it never rebuilds it.


In [ ]:
# PYTHON CELL
from google.colab import files

AUTO_DOWNLOAD = globals().get("AUTO_DOWNLOAD", True)
BUNDLE_DRIVE_PATH = RESULTS_DRIVE / f"score_pool_410m_{EXPERIMENT}_figure_bundle.zip"
SCORE_POOL_COLAB.verify_bundle(BUNDLE_DRIVE_PATH)
print("verified bundle:", BUNDLE_DRIVE_PATH, f"{BUNDLE_DRIVE_PATH.stat().st_size / 1_000_000:.1f} MB")
if AUTO_DOWNLOAD:
    files.download(str(BUNDLE_DRIVE_PATH))
print("Drive fallback:", BUNDLE_DRIVE_PATH)
